In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

PRICE_SOURCE_TZ = "America/New_York"
ANALYSIS_TZ = "Europe/Amsterdam"

# Load minute data

In [2]:
DATA_DIR = Path("Data/EURUSD 1m")
YEARS_TO_LOAD = range(2009, 2026)

all_price_files = sorted(DATA_DIR.glob("DAT_ASCII_EURUSD_M1_*.csv"))
selected_years = {str(year) for year in YEARS_TO_LOAD}
price_files = [path for path in all_price_files if path.stem.split("_")[-1] in selected_years]

column_names = ["timestamp_raw", "open", "high", "low", "close"]
dtype_map = {
    "open": "float32",
    "high": "float32",
    "low": "float32",
    "close": "float32",
}

frames = []
for price_file in price_files:
    year_df = pd.read_csv(
        price_file,
        sep=";",
        header=None,
        usecols=range(len(column_names)),
        names=column_names,
        dtype=dtype_map,
    )
    year_df["timestamp"] = (
        pd.to_datetime(year_df["timestamp_raw"], format="%Y%m%d %H%M%S", errors="raise")
        .dt.tz_localize(PRICE_SOURCE_TZ)
        .dt.tz_convert(ANALYSIS_TZ)
    )
    frames.append(year_df.drop(columns="timestamp_raw"))

eurusd = (
    pd.concat(frames, ignore_index=True)
    .sort_values("timestamp", kind="stable")
    .reset_index(drop=True)
)

loaded_years = [path.stem.split("_")[-1] for path in price_files]

print(f"Loaded {len(eurusd):,} rows from {len(price_files)} files.")
print(f"Years loaded: {loaded_years[0]} -> {loaded_years[-1]}")
print(f"Date range: {eurusd['timestamp'].min()} -> {eurusd['timestamp'].max()}")
display(eurusd.head())


Loaded 6,219,282 rows from 17 files.
Years loaded: 2009 -> 2025
Date range: 2009-01-02 00:59:00+01:00 -> 2025-12-31 22:57:00+01:00


,open,high,low,close,timestamp
0,1.4005,1.4005,1.4005,1.4005,2009-01-02 00:59:00+01:00
1,1.4008,1.4008,1.4003,1.4004,2009-01-02 01:00:00+01:00
2,1.4005,1.4009,1.4005,1.4008,2009-01-02 01:01:00+01:00
3,1.4009,1.4011,1.4009,1.4011,2009-01-02 01:02:00+01:00
4,1.4009,1.4009,1.4005,1.4005,2009-01-02 01:03:00+01:00


# Load event data

In [3]:
EVENT_FILES = {
    # "EUR": Path("Data/eur_events_backup.xlsx"),
    "US": Path("Data/us_events_backup.xlsx"),
}

event_frames = []
for region, workbook_path in EVENT_FILES.items():
    workbook = pd.read_excel(
        workbook_path,
        sheet_name=None,
        usecols="A:D",
        engine="openpyxl",
        na_values=["#N/A N/A"],
    )
    for event_name, event_df in workbook.items():
        event_df = event_df.copy()
        event_df.columns = ["date", "survey", "actual", "datetime_raw"]
        event_df["region"] = region
        event_df["event_name"] = event_name
        event_frames.append(event_df)

events = pd.concat(event_frames, ignore_index=True)
events["date"] = pd.to_datetime(events["date"], errors="coerce")
events["datetime_raw"] = pd.to_datetime(events["datetime_raw"], errors="coerce")

events["survey"] = pd.to_numeric(events["survey"], errors="coerce")
events["actual"] = pd.to_numeric(events["actual"], errors="coerce")

event_row_mask = events[["date", "survey", "actual", "datetime_raw"]].notna().any(axis=1)
helper_only_rows_dropped = int((~event_row_mask).sum())
events = events.loc[event_row_mask].copy()

events["datetime"] = (
    events["datetime_raw"]
    .dt.tz_localize(ANALYSIS_TZ, ambiguous="NaT", nonexistent="shift_forward")
)

events = (
    events.sort_values(["datetime", "region", "event_name"], kind="stable")
    .reset_index(drop=True)
)

# Filter for events after 2008
cutoff = pd.Timestamp('2009-01-01', tz=ANALYSIS_TZ)
events = events[events['datetime'] >= cutoff]

# filter for specific event
# events = events[events['event_name'] == 'bus_cond']

print(f"Loaded {len(events):,} event rows from {len(EVENT_FILES)} workbooks.")
print(f"Date range: {events['datetime'].min()} -> {events['datetime'].max()}")
display(events.head())


Loaded 1,032 event rows from 1 workbooks.
Date range: 2009-01-02 16:00:00+01:00 -> 2026-03-18 19:00:00+01:00


,date,survey,actual,datetime_raw,region,event_name,datetime
512,2009-01-02,35.40,32.40,2009-01-02 16:00:00,US,bus_cond,2009-01-02 16:00:00+01:00
513,2009-01-09,-525.00,-524.00,2009-01-09 14:30:00,US,NFP,2009-01-09 14:30:00+01:00
514,2009-01-09,7.00,7.20,2009-01-09 14:30:00,US,unempl_rate,2009-01-09 14:30:00+01:00
515,2009-01-16,-0.20,0.10,2009-01-16 14:30:00,US,cpi_yoy,2009-01-16 14:30:00+01:00
516,2009-01-28,0.25,0.25,2009-01-28 20:15:00,US,cb_target_rate,2009-01-28 20:15:00+01:00


In [4]:
events_sign = events.copy()

events_sign["surprise"] = events_sign["actual"] - events_sign["survey"]
events_sign["surprise_sign"] = np.sign(events_sign["surprise"])

events_sign = events_sign.loc[events_sign["surprise_sign"] != 0].copy()
events_sign = events_sign.sort_values("datetime", kind="stable").reset_index(drop=True)

display(events_sign[["datetime", "region", "event_name", "survey", "actual", "surprise", "surprise_sign"]].head())

,datetime,region,event_name,survey,actual,surprise,surprise_sign
0,2009-01-02 16:00:00+01:00,US,bus_cond,35.4,32.4,-3.0,-1.0
1,2009-01-09 14:30:00+01:00,US,NFP,-525.0,-524.0,1.0,1.0
2,2009-01-09 14:30:00+01:00,US,unempl_rate,7.0,7.2,0.2,1.0
3,2009-01-16 14:30:00+01:00,US,cpi_yoy,-0.2,0.1,0.3,1.0
4,2009-02-02 16:00:00+01:00,US,bus_cond,32.5,35.6,3.1,1.0


In [5]:
price_lookup = (
    eurusd[["timestamp", "close"]]
    .dropna(subset=["timestamp", "close"])
    .sort_values("timestamp", kind="stable")
    .drop_duplicates(subset=["timestamp"])
    .set_index("timestamp")["close"]
)

event_moves = (
    events_sign[["datetime", "region", "event_name", "surprise", "surprise_sign"]]
    .dropna(subset=["datetime"])
    .sort_values("datetime", kind="stable")
    .reset_index(drop=True)
    .copy()
)

# use previous minute close as price_pre
event_moves["price_pre"] = price_lookup.reindex(
    event_moves["datetime"] - pd.Timedelta(minutes=1),
    method="ffill"
).to_numpy()

for mins in [1, 5, 10, 30]:
    target_times = event_moves["datetime"] + pd.Timedelta(minutes=mins - 1)
    event_moves[f"price_t{mins}"] = price_lookup.reindex(target_times, method="ffill").to_numpy()
    event_moves[f"ret_{mins}m"] = event_moves[f"price_t{mins}"] / event_moves["price_pre"] - 1
    event_moves[f"ret_sign_{mins}m"] = np.sign(event_moves[f"ret_{mins}m"])

display(event_moves.head())

,datetime,region,event_name,surprise,surprise_sign,price_pre,price_t1,ret_1m,ret_sign_1m,price_t5,ret_5m,ret_sign_5m,price_t10,ret_10m,ret_sign_10m,price_t30,ret_30m,ret_sign_30m
0,2009-01-02 16:00:00+01:00,US,bus_cond,-3.0,-1.0,1.3916,1.3921,0.000359,1.0,1.3943,0.001940,1.0,1.3946,0.002156,1.0,1.3974,0.004168,1.0
1,2009-01-09 14:30:00+01:00,US,NFP,1.0,1.0,1.3707,1.3740,0.002408,1.0,1.3696,-0.000802,-1.0,1.3711,0.000292,1.0,1.3609,-0.007150,-1.0
2,2009-01-09 14:30:00+01:00,US,unempl_rate,0.2,1.0,1.3707,1.3740,0.002408,1.0,1.3696,-0.000802,-1.0,1.3711,0.000292,1.0,1.3609,-0.007150,-1.0
3,2009-01-16 14:30:00+01:00,US,cpi_yoy,0.3,1.0,1.3278,1.3281,0.000226,1.0,1.3297,0.001431,1.0,1.3285,0.000527,1.0,1.3315,0.002787,1.0
4,2009-02-02 16:00:00+01:00,US,bus_cond,3.1,1.0,1.2793,1.2798,0.000391,1.0,1.2796,0.000234,1.0,1.2828,0.002736,1.0,1.2824,0.002423,1.0


In [6]:
summary_rows = []

for event_type in sorted(event_moves["event_name"].dropna().unique()):
    sub = event_moves.loc[event_moves["event_name"] == event_type].copy()

    for mins in [1, 5, 10, 30]:
        df = sub.loc[
            sub[f"ret_sign_{mins}m"].notna() & (sub[f"ret_sign_{mins}m"] != 0),
            ["surprise_sign", f"ret_sign_{mins}m"]
        ].copy()

        summary_rows.append({
            "event_type": event_type,
            "window_min": mins,
            "n_events": len(df),
            "same_sign_share": (df["surprise_sign"] == df[f"ret_sign_{mins}m"]).mean() if len(df) else np.nan,
        })

sign_matrix = pd.DataFrame(summary_rows).pivot(
    index="event_type",
    columns="window_min",
    values="same_sign_share"
).sort_index(axis=1)

display(sign_matrix)

window_min,1,5,10,30
event_type,,,,
NFP,0.306533,0.303030,0.300000,0.340000
bus_cond,0.319797,0.331658,0.360000,0.388889
cb_target_rate,0.200000,0.000000,0.000000,0.000000
cpi_yoy,0.204380,0.289855,0.333333,0.355072
gdp,0.326923,0.320755,0.313725,0.326923
unempl_rate,0.513514,0.560811,0.543624,0.550336


In [7]:
# check why CB behaves so nicely : )

display(
    pd.DataFrame(summary_rows).pivot(
        index="event_type",
        columns="window_min",
        values="n_events"
    ).sort_index(axis=1)
)

window_min,1,5,10,30
event_type,,,,
NFP,199,198,200,200
bus_cond,197,199,200,198
cb_target_rate,5,5,5,5
cpi_yoy,137,138,138,138
gdp,52,53,51,52
unempl_rate,148,148,149,149
